## Montar Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = '/content/drive/MyDrive/Meli_Case'

In [3]:
import sys
sys.path.append(path)

## Imports

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
from scipy.stats import ks_2samp
import shap
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler

## Funções

In [6]:
from functions import calculate_psi, analisar_metricas_por_decil

## Base

In [7]:
base = pd.read_excel(f"{path}/fraud_dataset.xlsx")
base = base.rename(columns={col: f"var_{col}" for col in base.columns if len(col) == 1 and col.isalpha()})

In [8]:
variaveis = ["var_Q", "var_R", "var_S", 'Monto']

for col in variaveis:
    base[col] = base[col].apply(lambda x: np.nan if isinstance(x, datetime) else x)

    base[col] = base[col].astype(str).str.replace(",", "").str.replace(" ", "").replace("None", np.nan)
    base[col] = pd.to_numeric(base[col], errors='coerce')

In [9]:
for col in ["var_C", "var_F", "var_G"]:
    base[f"{col}_log"] = np.log1p(base[col])

In [10]:
def mapear_pais(pais):
    if pais in ['BR', 'AR', 'MX']:
        return pais
    elif pais in ['ES', 'US', 'UY']:
        return 'ES_US_UY'
    else:
        return 'OUTROS'

base['pais_agg'] = base['var_J'].apply(mapear_pais)

In [11]:
pais_agg_dummies = pd.get_dummies(base['pais_agg'], prefix='pais_agg')
pais_agg_dummies = pais_agg_dummies.astype(int)

In [12]:
base = pd.concat([base, pais_agg_dummies], axis=1)
base

,var_A,var_B,var_C,var_D,var_E,var_F,var_G,var_H,var_I,var_J,...,Fraude,var_C_log,var_F_log,var_G_log,pais_agg,pais_agg_AR,pais_agg_BR,pais_agg_ES_US_UY,pais_agg_MX,pais_agg_OUTROS
0,0,10,50257.0,0,0,0.0,0.0,0,0,UY,...,1,10.824925,0.000000,0.0,ES_US_UY,0,0,1,0,0
1,0,10,29014.0,0,0,0.0,0.0,0,0,UY,...,1,10.275568,0.000000,0.0,ES_US_UY,0,0,1,0,0
2,0,7,92.0,0,1,0.0,0.0,0,1,UY,...,1,4.532599,0.000000,0.0,ES_US_UY,0,0,1,0,0
3,9,16,50269.0,0,0,0.0,0.0,0,0,UY,...,1,10.825164,0.000000,0.0,ES_US_UY,0,0,1,0,0
4,0,8,8180.0,0,0,0.0,0.0,0,0,UY,...,1,9.009570,0.000000,0.0,ES_US_UY,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16875,0,3,63302.0,0,1,0.5,0.0,0,0,BR,...,1,11.055688,0.405465,0.0,BR,0,1,0,0,0
16876,0,12,825.0,0,0,0.0,0.0,0,0,BR,...,1,6.716595,0.000000,0.0,BR,0,1,0,0,0
16877,1,3,81067.0,0,0,0.0,0.0,0,0,BR,...,1,11.303044,0.000000,0.0,BR,0,1,0,0,0
16878,0,9,398372.0,0,0,0.0,0.0,0,0,BR,...,1,12.895144,0.000000,0.0,BR,0,1,0,0,0


In [13]:
base['var_K_100'] = (base['var_K'] * 100).round()

In [14]:
drop_col = ['var_C', 'var_F', 'var_G', 'var_J', 'pais_agg', 'var_K']
base = base.drop(columns=drop_col, errors='ignore')
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16880 entries, 0 to 16879
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   var_A              16880 non-null  int64  
 1   var_B              16880 non-null  int64  
 2   var_D              16880 non-null  int64  
 3   var_E              16880 non-null  int64  
 4   var_H              16880 non-null  int64  
 5   var_I              16880 non-null  int64  
 6   var_L              16880 non-null  int64  
 7   var_M              16880 non-null  int64  
 8   var_N              16880 non-null  int64  
 9   var_O              16880 non-null  int64  
 10  var_P              16880 non-null  int64  
 11  var_Q              16863 non-null  float64
 12  var_R              16877 non-null  float64
 13  var_S              15575 non-null  float64
 14  Monto              16413 non-null  float64
 15  Fraude             16880 non-null  int64  
 16  var_C_log          136

In [15]:
df = base[base['Monto'].notna()]

## Base

In [16]:
base = pd.read_excel(f"{path}/fraud_dataset.xlsx")
base = base.rename(columns={col: f"var_{col}" for col in base.columns if len(col) == 1 and col.isalpha()})

In [17]:
variaveis = ["var_Q", "var_R", "var_S", 'Monto']

for col in variaveis:
    base[col] = base[col].apply(lambda x: np.nan if isinstance(x, datetime) else x)

    base[col] = base[col].astype(str).str.replace(",", "").str.replace(" ", "").replace("None", np.nan)
    base[col] = pd.to_numeric(base[col], errors='coerce')

In [18]:
for col in ["var_C", "var_F", "var_G"]:
    base[f"{col}_log"] = np.log1p(base[col])

In [19]:
def mapear_pais(pais):
    if pais in ['BR', 'AR', 'MX']:
        return pais
    elif pais in ['ES', 'US', 'UY']:
        return 'ES_US_UY'
    else:
        return 'OUTROS'

base['pais_agg'] = base['var_J'].apply(mapear_pais)

In [20]:
pais_agg_dummies = pd.get_dummies(base['pais_agg'], prefix='pais_agg')
pais_agg_dummies = pais_agg_dummies.astype(int)

In [21]:
base = pd.concat([base, pais_agg_dummies], axis=1)
base

,var_A,var_B,var_C,var_D,var_E,var_F,var_G,var_H,var_I,var_J,...,Fraude,var_C_log,var_F_log,var_G_log,pais_agg,pais_agg_AR,pais_agg_BR,pais_agg_ES_US_UY,pais_agg_MX,pais_agg_OUTROS
0,0,10,50257.0,0,0,0.0,0.0,0,0,UY,...,1,10.824925,0.000000,0.0,ES_US_UY,0,0,1,0,0
1,0,10,29014.0,0,0,0.0,0.0,0,0,UY,...,1,10.275568,0.000000,0.0,ES_US_UY,0,0,1,0,0
2,0,7,92.0,0,1,0.0,0.0,0,1,UY,...,1,4.532599,0.000000,0.0,ES_US_UY,0,0,1,0,0
3,9,16,50269.0,0,0,0.0,0.0,0,0,UY,...,1,10.825164,0.000000,0.0,ES_US_UY,0,0,1,0,0
4,0,8,8180.0,0,0,0.0,0.0,0,0,UY,...,1,9.009570,0.000000,0.0,ES_US_UY,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16875,0,3,63302.0,0,1,0.5,0.0,0,0,BR,...,1,11.055688,0.405465,0.0,BR,0,1,0,0,0
16876,0,12,825.0,0,0,0.0,0.0,0,0,BR,...,1,6.716595,0.000000,0.0,BR,0,1,0,0,0
16877,1,3,81067.0,0,0,0.0,0.0,0,0,BR,...,1,11.303044,0.000000,0.0,BR,0,1,0,0,0
16878,0,9,398372.0,0,0,0.0,0.0,0,0,BR,...,1,12.895144,0.000000,0.0,BR,0,1,0,0,0


In [22]:
base['var_K_100'] = (base['var_K'] * 100).round()

In [23]:
drop_col = ['var_C', 'var_F', 'var_G', 'var_J', 'pais_agg', 'var_K']
base = base.drop(columns=drop_col, errors='ignore')
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16880 entries, 0 to 16879
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   var_A              16880 non-null  int64  
 1   var_B              16880 non-null  int64  
 2   var_D              16880 non-null  int64  
 3   var_E              16880 non-null  int64  
 4   var_H              16880 non-null  int64  
 5   var_I              16880 non-null  int64  
 6   var_L              16880 non-null  int64  
 7   var_M              16880 non-null  int64  
 8   var_N              16880 non-null  int64  
 9   var_O              16880 non-null  int64  
 10  var_P              16880 non-null  int64  
 11  var_Q              16863 non-null  float64
 12  var_R              16877 non-null  float64
 13  var_S              15575 non-null  float64
 14  Monto              16413 non-null  float64
 15  Fraude             16880 non-null  int64  
 16  var_C_log          136

In [24]:
df = base[base['Monto'].notna()]

## Treino, Teste e Validação

In [25]:
rf_fs = ['var_S',
 'Monto',
 'var_C_log',
 'var_B',
 'pais_agg_ES_US_UY',
 'var_K_100',
 'var_P',
 'var_M',
 'var_L',
 'var_A',
 'var_E',
 'pais_agg_MX',
 'var_F_log',
 'pais_agg_AR',
 'pais_agg_BR',
 'var_Q',
 'var_D']

In [26]:
X = df[rf_fs]
y = df["Fraude"]

In [27]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=13)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=13)

In [28]:
X_train.shape, X_test.shape, X_val.shape

((9847, 17), (3283, 17), (3283, 17))

## Monitormento de Variáveis utilizando o PSI

In [29]:
psi_teste = {var: calculate_psi(X_train[var].dropna(), X_test[var].dropna()) for var in rf_fs}
psi_val = {var: calculate_psi(X_train[var].dropna(), X_val[var].dropna()) for var in rf_fs}

In [30]:
psi_df = pd.DataFrame({'Teste': psi_teste, 'Validacao': psi_val})
psi_df.sort_values('Teste', ascending=False)

,Teste,Validacao
pais_agg_MX,0.0156,0.0006
pais_agg_AR,0.0098,0.0080
var_F_log,0.0047,0.0058
var_C_log,0.0041,0.0014
var_L,0.0035,0.0105
var_M,0.0024,0.0036
var_A,0.0021,0.0065
var_P,0.0018,0.0018
var_E,0.0016,0.0050
var_D,0.0016,0.0011


## Monitoramento de Falsos Positivos

In [31]:
best = {'colsample_bytree': np.float64(0.6406986982852445),
 'gamma': np.float64(0.7006462287403816),
 'learning_rate': np.float64(0.06789620347195029),
 'max_depth': np.float64(10.0),
 'min_child_weight': np.float64(7.0),
 'n_estimators': np.float64(230.0),
 'reg_alpha': np.float64(1.0503366753204815),
 'reg_lambda': np.float64(3.379213209035674),
 'subsample': np.float64(0.8219521946421204)}

In [32]:
final_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=13,
    n_jobs=-1,
    max_depth=int(best['max_depth']),
    learning_rate=best['learning_rate'],
    n_estimators=int(best['n_estimators']),
    gamma=best['gamma'],
    min_child_weight=int(best['min_child_weight']),
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda']
)

final_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=np.float64(0.6406986982852445), device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None,
              gamma=np.float64(0.7006462287403816), grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=np.float64(0.06789620347195029), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=7, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=230, n_jobs=-1,
              num_parallel_tree=None, random_state=13, ...)

In [33]:
y_probs_test = final_model.predict_proba(X_test)[:, 1]
y_probs_val = final_model.predict_proba(X_val)[:, 1]

In [34]:
best_threshold_f1 = 0.33

In [35]:
y_pred_test = (y_probs_test > best_threshold_f1).astype(int)
y_pred_val = (y_probs_val > best_threshold_f1).astype(int)

In [36]:
analisar_metricas_por_decil(y_val, y_probs_val)

,Decil,Score Mínimo,Score Máximo,Transações,Fraudes,Hit Rate,Detection Rate,Alert Rate
0,10,653,994,329,279,0.8480,0.3103,0.1002
1,9,462,653,328,186,0.5671,0.2069,0.0999
2,8,332,462,328,128,0.3902,0.1424,0.0999
3,7,238,332,328,104,0.3171,0.1157,0.0999
4,6,173,237,328,75,0.2287,0.0834,0.0999
5,5,122,173,329,32,0.0973,0.0356,0.1002
6,4,83,122,328,46,0.1402,0.0512,0.0999
7,3,56,83,328,25,0.0762,0.0278,0.0999
8,2,30,56,328,15,0.0457,0.0167,0.0999
9,1,6,30,329,9,0.0274,0.0100,0.1002


In [37]:

decil_10_df = analisar_metricas_por_decil(y_val, y_probs_val)
decil_10_min_score = decil_10_df.iloc[0]['Score Mínimo'] / 1000
decil_10_max_score = decil_10_df.iloc[0]['Score Máximo'] / 1000

In [38]:
df_val = pd.concat([X_val, y_val], axis=1)
df_val['Fraude_prob'] = y_probs_val
df_val['Fraude_pred'] = y_pred_val
df_val

,var_S,Monto,var_C_log,var_B,pais_agg_ES_US_UY,var_K_100,var_P,var_M,var_L,var_A,var_E,pais_agg_MX,var_F_log,pais_agg_AR,pais_agg_BR,var_Q,var_D,Fraude,Fraude_prob,Fraude_pred
16153,65.12,920.50,10.453745,7,0,69.0,3,2,2,4,0,0,0.0,0,1,0.00,0,1,0.639887,1
712,NaN,224.38,10.131937,12,0,NaN,1,2,0,2,0,1,0.0,0,0,0.00,0,1,0.269964,0
2242,45.56,60.62,6.086775,15,0,NaN,2,2,1,0,0,0,0.0,1,0,62.68,0,1,0.492269,1
7312,1.55,92.50,NaN,1,0,85.0,4,3,0,0,1,0,0.0,1,0,0.00,0,0,0.061450,0
7211,46.43,42.36,5.583496,13,0,50.0,1,1,0,0,0,0,0.0,1,0,0.00,0,0,0.402726,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9799,NaN,289.91,NaN,9,0,73.0,5,5,0,0,0,0,0.0,1,0,0.00,0,0,0.333605,1
11002,1.27,21.17,9.473627,2,1,NaN,1,1,0,0,0,0,0.0,0,0,0.00,0,0,0.787264,1
4511,0.75,8.96,NaN,1,0,NaN,2,2,0,0,0,0,0.0,1,0,0.00,0,0,0.019278,0
306,36.78,312.53,9.749870,6,1,NaN,1,1,1,1,0,0,0.0,0,0,0.00,0,1,0.942503,1


In [39]:
D10_NFraude = df_val[(df_val['Fraude'] == 0) & (df_val['Fraude_prob'] >= decil_10_min_score) & (df_val['Fraude_prob'] <= decil_10_max_score)]
D10_NFraude

,var_S,Monto,var_C_log,var_B,pais_agg_ES_US_UY,var_K_100,var_P,var_M,var_L,var_A,var_E,pais_agg_MX,var_F_log,pais_agg_AR,pais_agg_BR,var_Q,var_D,Fraude,Fraude_prob,Fraude_pred
10067,33.64,20.16,9.341456,14,0,57.0,4,2,1,0,0,0,0.000000,1,0,0.00,0,0,0.662127,1
11533,NaN,24.43,9.782393,1,1,NaN,1,1,2,0,0,0,0.000000,0,0,27.18,0,0,0.758044,1
7492,19.92,83.76,9.600963,10,0,NaN,5,3,0,0,0,0,0.000000,1,0,0.00,0,0,0.737764,1
11369,6.63,63.62,12.805434,4,1,NaN,1,1,1,0,0,0,0.000000,0,0,0.00,0,0,0.890958,1
3175,18.13,36.00,9.523763,12,0,NaN,5,3,0,0,0,0,0.000000,1,0,0.00,0,0,0.762295,1
8327,58.96,45.41,9.072112,11,0,64.0,5,3,0,0,0,0,0.000000,1,0,0.00,0,0,0.729482,1
7807,27.84,100.67,1.098612,9,0,NaN,1,1,1,1,0,0,0.000000,1,0,0.00,0,0,0.699897,1
5483,98.81,132.80,11.539957,10,0,NaN,1,1,0,1,0,0,0.000000,1,0,0.00,0,0,0.873842,1
4357,95.18,191.37,9.679594,10,0,NaN,2,1,0,2,0,0,0.000000,1,0,0.00,0,0,0.739119,1
6058,42.65,111.32,0.000000,11,0,NaN,2,2,0,0,0,0,0.000000,1,0,0.00,0,0,0.662071,1


## Salvar

In [40]:
# ! jupyter nbconvert --to html 03_Monitoramento.ipynb